In [ ]:
# Environment sanity check

from RL_evaluation_wrapper import ImprovedOT2Env

env = ImprovedOT2Env()

obs, info = env.reset()
print("Observation shape:", obs.shape)
print("Observation dtype:", obs.dtype)

assert obs.shape == (9,), "Observation shape mismatch"
assert obs.dtype == obs.dtype == obs.dtype, "Observation dtype mismatch"

# Single random step
import numpy as np
action = env.action_space.sample()
obs, reward, terminated, truncated, info = env.step(action)

print("Reward:", reward)
print("Terminated:", terminated, "Truncated:", truncated)

Observation shape: (9,)
Observation dtype: float32
Reward: -23.230468380451203
Terminated: False Truncated: False


pybullet build time: Jan 13 2026 19:10:51
/usr/local/lib/python3.12/dist-packages/gymnasium/spaces/box.py:236: UserWarning: WARN: Box low's precision lowered by casting to float32, current low.dtype=float64
  gym.logger.warn(
/usr/local/lib/python3.12/dist-packages/gymnasium/spaces/box.py:306: UserWarning: WARN: Box high's precision lowered by casting to float32, current high.dtype=float64
  gym.logger.warn(


In [2]:
# SB3 environment validation

from stable_baselines3.common.env_checker import check_env

check_env(env, warn=True)

2026-01-15 11:38:24.492562: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:485] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2026-01-15 11:38:24.514486: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:8473] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2026-01-15 11:38:24.521106: E external/local_xla/xla/stream_executor/cuda/cuda_blas.cc:1471] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2026-01-15 11:38:24.539541: I tensorflow/core/platform/cpu_feature_guard.cc:211] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: SSE3 SSE4.1 SSE4.2 AVX, in other operations, rebuild TensorFlow with the appropriate compiler flags.
/usr/local/lib/python3.12/dist-packages

In [3]:
# Load model

from stable_baselines3 import PPO
import os

MODEL_PATH = "ot2_ppo_improved_final.zip" 
model = PPO.load(MODEL_PATH)

/usr/local/lib/python3.12/dist-packages/stable_baselines3/common/save_util.py:167: UserWarning: Could not deserialize object clip_range. Consider using `custom_objects` argument to replace this object.
Exception: code() argument 13 must be str, not int
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/stable_baselines3/common/save_util.py:167: UserWarning: Could not deserialize object lr_schedule. Consider using `custom_objects` argument to replace this object.
Exception: code() argument 13 must be str, not int
  warnings.warn(


In [4]:
import numpy as np
import time

NUM_TARGETS = 83
SUCCESS_THRESHOLD = 0.001  # 1 mm
SIM_DT = 0.02  # seconds per step (adjust if needed)

results = {
    "success": [],
    "final_error_mm": [],
    "time_sec": []
}

start_time = time.time()

for i in range(NUM_TARGETS):
    obs, _ = env.reset()
    
    steps = 0
    
    done = False
    while not done:
        action, _ = model.predict(obs, deterministic=True)
        obs, reward, terminated, truncated, info = env.step(action)
        done = terminated or truncated


    final_pos = env.get_true_position()
    final_error = np.linalg.norm(final_pos - env.target)

    results["success"].append(final_error < SUCCESS_THRESHOLD)
    results["final_error_mm"].append(final_error * 1000)
    results["time_sec"].append(steps * SIM_DT)

total_time = time.time() - start_time

In [14]:
print("RL Controller Evaluation Summary:")
print(f"  Total targets: {NUM_TARGETS}")
print(f"  Successfully reached: {sum(results['success'])}")
print(f"  Success rate: {np.mean(results['success']) * 100:.1f}%\n")

print("Positioning accuracy:")
print(f"  Mean error: {np.mean(results['final_error_mm']):.2f} mm")
print(f"  Std error:  {np.std(results['final_error_mm']):.2f} mm")
print(f"  Max error:  {np.max(results['final_error_mm']):.2f} mm\n")

print("Execution time:")
print(f"  Mean time: {(total_time/NUM_TARGETS):.1f} sec/plant")
print(f"  Total time: {total_time:.1f} sec")


RL Controller Evaluation Summary:
  Total targets: 83
  Successfully reached: 0
  Success rate: 0.0%
 never within 1mm
Positioning accuracy:
  Mean error: 2.83 mm
  Std error:  0.23 mm
  Max error:  3.53 mm

Execution time:
  Mean time: 0.3 sec/plant
  Total time: 21.9 sec
